# Notebook 10: Batch 3 Inventory and Inspection

## Objective

Inspect the newly extracted PANORAMA Batch 3 data before preprocessing.

This notebook will:

- identify Batch 3 CT cases
- exclude the held-out `100936_00001` case
- inspect CT file structure
- inspect automatic and manual label availability
- check basic file readability
- compare CT and selected-label geometry
- produce a final Batch 3 eligibility inventory

No preprocessing will be performed in this notebook.

In [8]:
# ============================================================
# NOTEBOOK 10 — BATCH 3 INVENTORY
# ============================================================

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import SimpleITK as sitk

# ------------------------------------------------------------
# Project root
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("=" * 70)
print("BATCH 3 INVENTORY — PROJECT SETUP")
print("=" * 70)

print("Project root:")
print(PROJECT_ROOT)

BATCH 3 INVENTORY — PROJECT SETUP
Project root:
D:\Pancreatic_Cancer_Thesis


In [9]:
# ============================================================
# DIRECTORY CONFIGURATION
# ============================================================

DATA_DIR = PROJECT_ROOT / "data"

RAW_CT_DIR = DATA_DIR / "raw_ct"
LABELS_DIR = DATA_DIR / "labels"

AUTO_LABEL_DIR = LABELS_DIR / "Automatic_Labels"
MANUAL_LABEL_DIR = LABELS_DIR / "Manual_Labels"

PROCESSED_DIR = DATA_DIR / "processed"

BATCH3_INVENTORY_FILE = (
    PROCESSED_DIR / "batch3_inventory.csv"
)

HELD_OUT_CASE = "100936_00001"

print("=" * 70)
print("DIRECTORIES")
print("=" * 70)

paths = {
    "Raw CT": RAW_CT_DIR,
    "Automatic labels": AUTO_LABEL_DIR,
    "Manual labels": MANUAL_LABEL_DIR,
    "Processed": PROCESSED_DIR,
    "Batch 3 inventory": BATCH3_INVENTORY_FILE,
}

for name, path in paths.items():
    print(f"{name:22s}: {path}")
    print(f"{'Exists':22s}: {path.exists()}")
    print()

DIRECTORIES
Raw CT                : D:\Pancreatic_Cancer_Thesis\data\raw_ct
Exists                : True

Automatic labels      : D:\Pancreatic_Cancer_Thesis\data\labels\Automatic_Labels
Exists                : True

Manual labels         : D:\Pancreatic_Cancer_Thesis\data\labels\Manual_Labels
Exists                : True

Processed             : D:\Pancreatic_Cancer_Thesis\data\processed
Exists                : True

Batch 3 inventory     : D:\Pancreatic_Cancer_Thesis\data\processed\batch3_inventory.csv
Exists                : False



In [6]:
# ============================================================
# CELL 4 — IDENTIFY BATCH 3 CT CASES
# ============================================================

print("=" * 70)
print("IDENTIFYING BATCH 3 CT CASES")
print("=" * 70)

ct_files = sorted(
    RAW_CT_DIR.glob("*_0000.nii.gz")
)

study_ids_all = sorted(
    p.name.replace("_0000.nii.gz", "")
    for p in ct_files
)

print("Total CT files currently in raw_ct:", len(study_ids_all))

# ------------------------------------------------------------
# The only non-Batch-3 CT intentionally kept in raw_ct is:
# 100936_00001
# ------------------------------------------------------------

batch3_candidate_ids = [
    study_id
    for study_id in study_ids_all
    if study_id != HELD_OUT_CASE
]

print("\nAll CT cases found:")
print(len(study_ids_all))

print("\nHeld-out case present:")
print(HELD_OUT_CASE in study_ids_all)

print("\nBatch 3 candidates:")
print(len(batch3_candidate_ids))

print("\nFirst 20 Batch 3 candidates:")
for study_id in batch3_candidate_ids[:20]:
    print(" ", study_id)

print("\n" + "-" * 70)

if HELD_OUT_CASE in batch3_candidate_ids:
    raise RuntimeError(
        f"{HELD_OUT_CASE} was not excluded from Batch 3."
    )

print("✓ Batch 3 candidate list created.")

IDENTIFYING BATCH 3 CT CASES
Total CT files currently in raw_ct: 581

All CT cases found:
581

Held-out case present:
True

Batch 3 candidates:
580

First 20 Batch 3 candidates:
  101112_00001
  101113_00001
  101114_00001
  101115_00001
  101116_00001
  101117_00001
  101118_00001
  101119_00001
  101120_00001
  101121_00001
  101122_00001
  101123_00001
  101124_00001
  101125_00001
  101126_00001
  101127_00001
  101128_00001
  101129_00001
  101130_00001
  101131_00001

----------------------------------------------------------------------
✓ Batch 3 candidate list created.


In [10]:
# ============================================================
# CELL 5 — CHECK AGAINST EXISTING PROCESSED DATASET
# ============================================================

print("=" * 70)
print("CHECKING AGAINST EXISTING PROCESSED DATASET")
print("=" * 70)

metadata_path = PROCESSED_DIR / "metadata.csv"

if not metadata_path.exists():
    raise FileNotFoundError(
        f"Processed metadata not found:\n{metadata_path}"
    )

processed_metadata = pd.read_csv(
    metadata_path
)

processed_ids = set(
    processed_metadata["study_id"]
    .astype(str)
    .str.strip()
)

batch3_set = set(batch3_candidate_ids)

already_processed = (
    batch3_set & processed_ids
)

new_batch3_candidates = (
    batch3_set - processed_ids
)

print("Batch 3 candidates       :", len(batch3_set))
print("Already in metadata      :", len(already_processed))
print("New relative to dataset :", len(new_batch3_candidates))

if already_processed:
    print("\nAlready processed cases:")
    for study_id in sorted(already_processed):
        print(" ", study_id)

print("\n✓ Comparison complete.")

CHECKING AGAINST EXISTING PROCESSED DATASET
Batch 3 candidates       : 580
Already in metadata      : 0
New relative to dataset : 580

✓ Comparison complete.


In [11]:
# ============================================================
# BATCH 3 CT FILE INVENTORY
# ============================================================

print("=" * 70)
print("BATCH 3 CT FILE INVENTORY")
print("=" * 70)

ct_inventory = []

for study_id in batch3_candidate_ids:

    ct_path = (
        RAW_CT_DIR
        / f"{study_id}_0000.nii.gz"
    )

    ct_inventory.append(
        {
            "study_id": study_id,
            "ct_exists": ct_path.exists(),
            "ct_path": str(ct_path),
            "ct_size_mb": (
                ct_path.stat().st_size / (1024 ** 2)
                if ct_path.exists()
                else np.nan
            ),
        }
    )

batch3_ct_inventory = pd.DataFrame(
    ct_inventory
)

print("Cases:", len(batch3_ct_inventory))

print(
    "\nMissing CT files:",
    (~batch3_ct_inventory["ct_exists"]).sum()
)

display(
    batch3_ct_inventory.head(20)
)

BATCH 3 CT FILE INVENTORY
Cases: 580

Missing CT files: 0


,study_id,ct_exists,ct_path,ct_size_mb
0,101112_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101112...,59.616468
1,101113_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101113...,34.659803
2,101114_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101114...,64.151885
3,101115_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101115...,68.714606
4,101116_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101116...,25.418695
5,101117_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101117...,182.193948
6,101118_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101118...,148.479609
7,101119_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101119...,20.697845
8,101120_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101120...,41.676370
9,101121_00001,True,D:\Pancreatic_Cancer_Thesis\data\raw_ct\101121...,56.850116


In [12]:
# ============================================================
# LABEL AVAILABILITY
# ============================================================

print("=" * 70)
print("BATCH 3 LABEL AVAILABILITY")
print("=" * 70)

records = []

for study_id in batch3_candidate_ids:

    automatic_path = (
        AUTO_LABEL_DIR
        / f"{study_id}.nii.gz"
    )

    manual_path = (
        MANUAL_LABEL_DIR
        / f"{study_id}.nii.gz"
    )

    automatic_exists = automatic_path.exists()
    manual_exists = manual_path.exists()

    records.append(
        {
            "study_id": study_id,
            "ct_exists": (
                RAW_CT_DIR
                / f"{study_id}_0000.nii.gz"
            ).exists(),
            "automatic_label_exists": automatic_exists,
            "manual_label_exists": manual_exists,
        }
    )

batch3_inventory = pd.DataFrame(records)

# ------------------------------------------------------------
# Useful label-status flags
# ------------------------------------------------------------

batch3_inventory["has_any_label"] = (
    batch3_inventory["automatic_label_exists"]
    |
    batch3_inventory["manual_label_exists"]
)

batch3_inventory["has_both_labels"] = (
    batch3_inventory["automatic_label_exists"]
    &
    batch3_inventory["manual_label_exists"]
)

batch3_inventory["has_automatic_only"] = (
    batch3_inventory["automatic_label_exists"]
    &
    ~batch3_inventory["manual_label_exists"]
)

batch3_inventory["has_manual_only"] = (
    batch3_inventory["manual_label_exists"]
    &
    ~batch3_inventory["automatic_label_exists"]
)

batch3_inventory["has_no_label"] = (
    ~batch3_inventory["automatic_label_exists"]
    &
    ~batch3_inventory["manual_label_exists"]
)


print(
    "Total Batch 3 candidates:",
    len(batch3_inventory)
)

print(
    "\nAutomatic labels:",
    int(
        batch3_inventory[
            "automatic_label_exists"
        ].sum()
    )
)

print(
    "Manual labels:",
    int(
        batch3_inventory[
            "manual_label_exists"
        ].sum()
    )
)

print(
    "Both labels:",
    int(
        batch3_inventory[
            "has_both_labels"
        ].sum()
    )
)

print(
    "Automatic only:",
    int(
        batch3_inventory[
            "has_automatic_only"
        ].sum()
    )
)

print(
    "Manual only:",
    int(
        batch3_inventory[
            "has_manual_only"
        ].sum()
    )
)

print(
    "No label:",
    int(
        batch3_inventory[
            "has_no_label"
        ].sum()
    )
)

BATCH 3 LABEL AVAILABILITY
Total Batch 3 candidates: 580

Automatic labels: 447
Manual labels: 133
Both labels: 0
Automatic only: 447
Manual only: 133
No label: 0


In [13]:
# ============================================================
# SELECT PREFERRED LABEL
# ============================================================

def select_label(row):
    if row["automatic_label_exists"]:
        return "automatic"
    elif row["manual_label_exists"]:
        return "manual"
    else:
        return None


batch3_inventory["selected_label_type"] = (
    batch3_inventory.apply(
        select_label,
        axis=1,
    )
)

batch3_inventory["selected_label_path"] = None

for idx, row in batch3_inventory.iterrows():

    study_id = row["study_id"]
    label_type = row["selected_label_type"]

    if label_type == "automatic":

        path = (
            AUTO_LABEL_DIR
            / f"{study_id}.nii.gz"
        )

    elif label_type == "manual":

        path = (
            MANUAL_LABEL_DIR
            / f"{study_id}.nii.gz"
        )

    else:

        path = None

    batch3_inventory.loc[
        idx,
        "selected_label_path"
    ] = (
        str(path)
        if path is not None
        else None
    )

print("=" * 70)
print("SELECTED LABEL SUMMARY")
print("=" * 70)

print(
    batch3_inventory[
        "selected_label_type"
    ].value_counts(
        dropna=False
    )
)

SELECTED LABEL SUMMARY
selected_label_type
automatic    447
manual       133
Name: count, dtype: int64


In [14]:
# ============================================================
# CT + SELECTED LABEL READABILITY
# ============================================================

print("=" * 70)
print("BATCH 3 READABILITY CHECK")
print("=" * 70)

results = []

for i, row in enumerate(
    batch3_inventory.itertuples(index=False),
    start=1,
):

    study_id = row.study_id

    ct_path = (
        RAW_CT_DIR
        / f"{study_id}_0000.nii.gz"
    )

    label_type = row.selected_label_type
    label_path = (
        Path(row.selected_label_path)
        if row.selected_label_path
        else None
    )

    result = {
        "study_id": study_id,
        "ct_readable": False,
        "label_readable": False,
        "ct_shape": None,
        "label_shape": None,
        "ct_spacing": None,
        "label_spacing": None,
        "ct_origin": None,
        "label_origin": None,
        "ct_direction": None,
        "label_direction": None,
        "error": None,
    }

    try:

        ct_image = sitk.ReadImage(
            str(ct_path)
        )

        result["ct_readable"] = True
        result["ct_shape"] = (
            tuple(ct_image.GetSize())
        )
        result["ct_spacing"] = (
            tuple(ct_image.GetSpacing())
        )
        result["ct_origin"] = (
            tuple(ct_image.GetOrigin())
        )
        result["ct_direction"] = (
            tuple(ct_image.GetDirection())
        )

        if label_path is None:

            raise FileNotFoundError(
                "No selected label."
            )

        label_image = sitk.ReadImage(
            str(label_path)
        )

        result["label_readable"] = True
        result["label_shape"] = (
            tuple(label_image.GetSize())
        )
        result["label_spacing"] = (
            tuple(label_image.GetSpacing())
        )
        result["label_origin"] = (
            tuple(label_image.GetOrigin())
        )
        result["label_direction"] = (
            tuple(label_image.GetDirection())
        )

    except Exception as e:

        result["error"] = str(e)

    results.append(result)

    if (
        i % 25 == 0
        or i == len(batch3_inventory)
    ):
        print(
            f"Checked {i}/{len(batch3_inventory)}"
        )


batch3_validation = pd.DataFrame(
    results
)

print("\n" + "-" * 70)

print(
    "CT readable:",
    int(
        batch3_validation[
            "ct_readable"
        ].sum()
    ),
    "/",
    len(batch3_validation),
)

print(
    "Label readable:",
    int(
        batch3_validation[
            "label_readable"
        ].sum()
    ),
    "/",
    len(batch3_validation),
)

BATCH 3 READABILITY CHECK
Checked 25/580
Checked 50/580
Checked 75/580
Checked 100/580
Checked 125/580
Checked 150/580
Checked 175/580
Checked 200/580
Checked 225/580
Checked 250/580
Checked 275/580
Checked 300/580
Checked 325/580
Checked 350/580
Checked 375/580
Checked 400/580
Checked 425/580
Checked 450/580
Checked 475/580
Checked 500/580
Checked 525/580
Checked 550/580
Checked 575/580
Checked 580/580

----------------------------------------------------------------------
CT readable: 580 / 580
Label readable: 580 / 580


In [15]:
# ============================================================
# CT ↔ SELECTED LABEL GEOMETRY
# ============================================================

print("=" * 70)
print("CT ↔ LABEL GEOMETRY VALIDATION")
print("=" * 70)

geometry_results = []

for _, row in batch3_validation.iterrows():

    result = {
        "study_id": row["study_id"],
        "shape_match": False,
        "spacing_match": False,
        "origin_match": False,
        "direction_match": False,
        "geometry_match": False,
        "error": row["error"],
    }

    if (
        not row["ct_readable"]
        or not row["label_readable"]
    ):
        geometry_results.append(result)
        continue

    result["shape_match"] = (
        row["ct_shape"] == row["label_shape"]
    )

    result["spacing_match"] = np.allclose(
        row["ct_spacing"],
        row["label_spacing"],
        atol=1e-5,
    )

    result["origin_match"] = np.allclose(
        row["ct_origin"],
        row["label_origin"],
        atol=1e-4,
    )

    result["direction_match"] = np.allclose(
        row["ct_direction"],
        row["label_direction"],
        atol=1e-5,
    )

    result["geometry_match"] = (
        result["shape_match"]
        and result["spacing_match"]
        and result["origin_match"]
        and result["direction_match"]
    )

    geometry_results.append(result)


batch3_geometry = pd.DataFrame(
    geometry_results
)

print(
    "Shape matching:",
    int(batch3_geometry["shape_match"].sum()),
    "/",
    len(batch3_geometry),
)

print(
    "Spacing matching:",
    int(batch3_geometry["spacing_match"].sum()),
    "/",
    len(batch3_geometry),
)

print(
    "Origin matching:",
    int(batch3_geometry["origin_match"].sum()),
    "/",
    len(batch3_geometry),
)

print(
    "Direction matching:",
    int(batch3_geometry["direction_match"].sum()),
    "/",
    len(batch3_geometry),
)

print(
    "Full geometry matching:",
    int(batch3_geometry["geometry_match"].sum()),
    "/",
    len(batch3_geometry),
)

CT ↔ LABEL GEOMETRY VALIDATION
Shape matching: 580 / 580
Spacing matching: 578 / 580
Origin matching: 580 / 580
Direction matching: 580 / 580
Full geometry matching: 578 / 580


In [17]:
# ============================================================
# BATCH 3 — INSPECT THE TWO SPACING MISMATCHES
# ============================================================

import numpy as np
import SimpleITK as sitk

problem_ids = [
    "101351_00001",
    "101475_00001",
]

print("=" * 70)
print("BATCH 3 SPACING MISMATCH INSPECTION")
print("=" * 70)

for study_id in problem_ids:

    print("\n" + "=" * 70)
    print("CASE:", study_id)
    print("=" * 70)

    # --------------------------------------------------------
    # Paths
    # --------------------------------------------------------

    ct_path = (
        RAW_CT_DIR
        / f"{study_id}_0000.nii.gz"
    )

    automatic_path = (
        AUTO_LABEL_DIR
        / f"{study_id}.nii.gz"
    )

    manual_path = (
        MANUAL_LABEL_DIR
        / f"{study_id}.nii.gz"
    )

    # Determine which label was selected
    row = batch3_inventory[
        batch3_inventory["study_id"] == study_id
    ].iloc[0]

    label_type = row["selected_label_type"]

    if label_type == "automatic":
        label_path = automatic_path
    elif label_type == "manual":
        label_path = manual_path
    else:
        raise RuntimeError(
            f"No selected label for {study_id}"
        )

    # --------------------------------------------------------
    # Read
    # --------------------------------------------------------

    ct = sitk.ReadImage(
        str(ct_path)
    )

    label = sitk.ReadImage(
        str(label_path)
    )

    # --------------------------------------------------------
    # Print geometry
    # --------------------------------------------------------

    print("\nSelected label type:")
    print(" ", label_type)

    print("\nCT")
    print("  Path      :", ct_path)
    print("  Size      :", ct.GetSize())
    print("  Spacing   :", ct.GetSpacing())
    print("  Origin    :", ct.GetOrigin())
    print("  Direction :", ct.GetDirection())

    print("\nLabel")
    print("  Path      :", label_path)
    print("  Size      :", label.GetSize())
    print("  Spacing   :", label.GetSpacing())
    print("  Origin    :", label.GetOrigin())
    print("  Direction :", label.GetDirection())

    # --------------------------------------------------------
    # Differences
    # --------------------------------------------------------

    spacing_difference = (
        np.asarray(label.GetSpacing())
        - np.asarray(ct.GetSpacing())
    )

    print("\nSpacing difference (label - CT):")
    print(" ", spacing_difference)

    print("\nGeometry:")
    print(
        "  Shape match    :",
        ct.GetSize() == label.GetSize()
    )

    print(
        "  Spacing match  :",
        np.allclose(
            ct.GetSpacing(),
            label.GetSpacing(),
            atol=1e-5,
        )
    )

    print(
        "  Origin match   :",
        np.allclose(
            ct.GetOrigin(),
            label.GetOrigin(),
            atol=1e-4,
        )
    )

    print(
        "  Direction match:",
        np.allclose(
            ct.GetDirection(),
            label.GetDirection(),
            atol=1e-5,
        )
    )

BATCH 3 SPACING MISMATCH INSPECTION

CASE: 101351_00001

Selected label type:
  manual

CT
  Path      : D:\Pancreatic_Cancer_Thesis\data\raw_ct\101351_00001_0000.nii.gz
  Size      : (512, 512, 934)
  Spacing   : (0.71875, 0.71875, 0.699999988079071)
  Origin    : (-190.640625, -334.640625, -669.5999755859375)
  Direction : (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)

Label
  Path      : D:\Pancreatic_Cancer_Thesis\data\labels\Manual_Labels\101351_00001.nii.gz
  Size      : (512, 512, 934)
  Spacing   : (0.71875, 0.71875, 0.699951171875)
  Origin    : (-190.640625, -334.640625, -669.5999755859375)
  Direction : (1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)

Spacing difference (label - CT):
  [ 0.00000000e+00  0.00000000e+00 -4.88162041e-05]

Geometry:
  Shape match    : True
  Spacing match  : False
  Origin match   : True
  Direction match: True

CASE: 101475_00001

Selected label type:
  manual

CT
  Path      : D:\Pancreatic_Cancer_Thesis\data\raw_ct\101475_00001_0000.nii.gz
  Si

In [18]:
# ============================================================
# BATCH 3 — RECHECK SPACING WITH PRACTICAL TOLERANCE
# ============================================================

SPACING_TOLERANCE = 1e-4

batch3_geometry["spacing_match"] = (
    batch3_validation.apply(
        lambda row: (
            row["ct_readable"]
            and row["label_readable"]
            and np.allclose(
                row["ct_spacing"],
                row["label_spacing"],
                atol=SPACING_TOLERANCE,
            )
        ),
        axis=1,
    )
)

batch3_geometry["geometry_match"] = (
    batch3_geometry["shape_match"]
    & batch3_geometry["spacing_match"]
    & batch3_geometry["origin_match"]
    & batch3_geometry["direction_match"]
)

print("=" * 70)
print("BATCH 3 GEOMETRY RECHECK")
print("=" * 70)

print(
    "Spacing matching:",
    int(batch3_geometry["spacing_match"].sum()),
    "/",
    len(batch3_geometry),
)

print(
    "Full geometry matching:",
    int(batch3_geometry["geometry_match"].sum()),
    "/",
    len(batch3_geometry),
)

print("\nCases still failing geometry:")

remaining = batch3_geometry[
    ~batch3_geometry["geometry_match"]
]

display(remaining)

BATCH 3 GEOMETRY RECHECK
Spacing matching: 580 / 580
Full geometry matching: 580 / 580

Cases still failing geometry:


,study_id,shape_match,spacing_match,origin_match,direction_match,geometry_match,error


In [19]:
# ============================================================
# BUILD FINAL BATCH 3 INVENTORY TABLE
# ============================================================

batch3_inventory = batch3_inventory.merge(
    batch3_validation,
    on="study_id",
    how="left",
)

batch3_inventory = batch3_inventory.merge(
    batch3_geometry[
        [
            "study_id",
            "shape_match",
            "spacing_match",
            "origin_match",
            "direction_match",
            "geometry_match",
        ]
    ],
    on="study_id",
    how="left",
)

print("=" * 70)
print("BATCH 3 INVENTORY TABLE")
print("=" * 70)

print("Shape:", batch3_inventory.shape)

display(
    batch3_inventory.head()
)

BATCH 3 INVENTORY TABLE
Shape: (580, 27)


,study_id,ct_exists,automatic_label_exists,manual_label_exists,has_any_label,has_both_labels,has_automatic_only,has_manual_only,has_no_label,selected_label_type,...,ct_origin,label_origin,ct_direction,label_direction,error,shape_match,spacing_match,origin_match,direction_match,geometry_match
0,101112_00001,True,True,False,True,False,True,False,False,automatic,...,"(-159.6875, -159.6875, 1267.0)","(-159.6875, -159.6875, 1267.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
1,101113_00001,True,True,False,True,False,True,False,False,automatic,...,"(-199.6092987060547, -199.6092987060547, 1509.5)","(-199.6092987060547, -199.6092987060547, 1509.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
2,101114_00001,True,True,False,True,False,True,False,False,automatic,...,"(-227.14805603027344, -199.80430603027344, 105...","(-227.14805603027344, -199.80430603027344, 105...","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
3,101115_00001,True,True,False,True,False,True,False,False,automatic,...,"(-186.646484375, -335.646484375, -515.5)","(-186.646484375, -335.646484375, -515.5)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)","(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True
4,101116_00001,True,True,False,True,False,True,False,False,automatic,...,"(329.3553466796875, 329.3553466796875, 0.0)","(329.3553466796875, 329.3553466796875, 0.0)","(-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)","(-1.0, 0.0, 0.0, 0.0, -1.0, 0.0, 0.0, 0.0, 1.0)",None,True,True,True,True,True


In [20]:
# ============================================================
# POTENTIALLY PROBLEMATIC BATCH 3 CASES
# ============================================================

problems = batch3_inventory[
    (~batch3_inventory["ct_readable"])
    | (~batch3_inventory["label_readable"])
    | (~batch3_inventory["geometry_match"])
].copy()

print("=" * 70)
print("POTENTIALLY PROBLEMATIC BATCH 3 CASES")
print("=" * 70)

print(
    "Cases requiring review:",
    len(problems)
)

if len(problems) > 0:
    display(
        problems[
            [
                "study_id",
                "automatic_label_exists",
                "manual_label_exists",
                "selected_label_type",
                "ct_readable",
                "label_readable",
                "shape_match",
                "spacing_match",
                "origin_match",
                "direction_match",
                "geometry_match",
                "error",
            ]
        ]
    )
else:
    print(
        "✓ No readability or geometry problems detected."
    )

POTENTIALLY PROBLEMATIC BATCH 3 CASES
Cases requiring review: 0
✓ No readability or geometry problems detected.


In [21]:
# ============================================================
# SAVE BATCH 3 INVENTORY
# ============================================================

BATCH3_INVENTORY_FILE.parent.mkdir(
    parents=True,
    exist_ok=True,
)

batch3_inventory.to_csv(
    BATCH3_INVENTORY_FILE,
    index=False,
)

print("=" * 70)
print("BATCH 3 INVENTORY SAVED")
print("=" * 70)

print(
    BATCH3_INVENTORY_FILE
)

print(
    "Rows:",
    len(batch3_inventory)
)

BATCH 3 INVENTORY SAVED
D:\Pancreatic_Cancer_Thesis\data\processed\batch3_inventory.csv
Rows: 580


In [22]:
# ============================================================
# FINAL BATCH 3 INVENTORY REPORT
# ============================================================

print("=" * 70)
print("FINAL BATCH 3 INVENTORY REPORT")
print("=" * 70)

total = len(batch3_inventory)

print(
    f"Batch 3 CT candidates        : {total}"
)

print(
    f"CT readable                  : "
    f"{int(batch3_inventory['ct_readable'].sum())}"
    f"/{total}"
)

print(
    f"Label readable              : "
    f"{int(batch3_inventory['label_readable'].sum())}"
    f"/{total}"
)

print(
    f"Geometry matching            : "
    f"{int(batch3_inventory['geometry_match'].sum())}"
    f"/{total}"
)

print("\n" + "-" * 70)

print("Selected labels:")

print(
    batch3_inventory[
        "selected_label_type"
    ].value_counts(
        dropna=False
    )
)

print("\n" + "-" * 70)

print(
    "Requires review:",
    len(problems)
)

print("=" * 70)

FINAL BATCH 3 INVENTORY REPORT
Batch 3 CT candidates        : 580
CT readable                  : 580/580
Label readable              : 580/580
Geometry matching            : 580/580

----------------------------------------------------------------------
Selected labels:
selected_label_type
automatic    447
manual       133
Name: count, dtype: int64

----------------------------------------------------------------------
Requires review: 0
